# Entity Linking and Information Retrieval Debugging with DeepFix

This tutorial demonstrates how to use DeepFix to diagnose and debug **Information Retrieval (IR)** and **Entity Linking** models. In these tasks:
- **Queries** (e.g., search queries or text mentions) are matched against a large corpus of **Documents** or **Entities**.
- The goal is to retrieve the most relevant documents/entities for each query.
- Evaluating these systems involves analyzing query-document pairs, relevance labels, and retrieval ranks/scores.

DeepFix provides native support for IR and Entity Linking datasets via the `InformationRetrievalDataset` class, allowing you to ingest your data, wrap your model with `IRLookupModel`, and get automated, detailed diagnostic reports (covering data drift, annotation quality, and retrieval performance).

In [ ]:
import os
import random
import pandas as pd
import numpy as np
import pyterrier as pt
import hashlib

from deepfix_sdk import DeepFixClient
from deepfix_sdk.data.datasets import InformationRetrievalDataset
from deepfix_sdk.models import IRLookupModel

In [ ]:
# Set up your DeepFix API key.
# Get your API key by signing up at https://deepfix.delcaux.com
os.environ["DEEPFIX_API_KEY"] = "your-api-key-here"

In [ ]:
# Initialize the DeepFix client
client = DeepFixClient(timeout=300)

## Loading and Preparing the Dataset

We will load a subset of the **BEIR scidocs** dataset using `pyterrier`.
To evaluate the retrieval model, we will:
1. Load queries (topics), ground truth relevance labels (qrels), and documents (corpus).
2. Generate deterministic random embeddings for queries and documents.
3. Simulate retrieval scores for a model to evaluate.

In [ ]:
def simulate_retrievals(qrels_df: pd.DataFrame, retrieval_rate: float = 0.8, seed: int = 42) -> pd.DataFrame:
    """Simulate model retrievals from a qrels DataFrame.
    
    Only a fraction of the actual relevant documents/entities are successfully retrieved.
    For retrieved items, we generate simulated confidence scores/probabilities for binary relevance classes.
    """
    rng = np.random.default_rng(seed)
    n = len(qrels_df)

    # Only a fraction of items are "found" by the model
    retrieved_mask = rng.random(n) < retrieval_rate
    df = qrels_df[retrieved_mask].copy()
    m = len(df)

    prob = rng.random(m)
    is_relevant = df["relevance"].values == 1
    # 80% of relevant docs get a high class-1 score, 20% get a low one
    high_class1 = rng.random(m) > 0.2

    score_class0 = np.where(
        is_relevant,
        np.where(high_class1, prob * 0.5, 1 - prob * 0.5),
        1 - prob * 0.2,
    )
    score_class1 = np.where(
        is_relevant,
        np.where(high_class1, 1 - prob * 0.5, prob * 0.5),
        prob * 0.2,
    )

    return pd.DataFrame(
        {
            "query_id": df["query_id"].values,
            "doc_id": df["doc_id"].values,
            "score": [[s0, s1] for s0, s1 in zip(score_class0, score_class1)],
            "relevance": (score_class1 > score_class0).astype(int),
            "rank": rng.integers(1, 101, size=m),
        }
    )


In [ ]:
def random_embedder(text: str, dim: int = 10) -> np.ndarray:
    """Compute a deterministic random embedding for a given text.
    
    Useful for demonstrating embedding-based validation (like embedding drift)
    without needing an external LLM or heavy deep learning library.
    """
    seed = int(hashlib.md5(text.encode()).hexdigest(), 16) % (2**32)
    rng = np.random.default_rng(seed)
    return rng.random(dim)

In [ ]:
def load_ir_data(subset_queries: int = 50):
    """Load BEIR dbpedia-entity data using PyTerrier and prepare IR datasets."""
    name = "irds:beir/scidocs"
    dataset = pt.get_dataset(name)

    # 1. Get all topics and qrels, subset for fast execution
    all_topics = dataset.get_topics()
    all_qrels = dataset.get_qrels()

    qid_subset = all_topics["qid"].unique()[:subset_queries]
    topics_df = all_topics[all_topics["qid"].isin(qid_subset)]
    qrels_df = all_qrels[all_qrels["qid"].isin(qid_subset)]

    # 2. Build a single dataset, then split using stratified sampling on labels
    ir_ds = InformationRetrievalDataset(
        dataset_name=name,
        topics=topics_df,
        qrels=qrels_df,
        corpus_iter=dataset.get_corpus_iter,
    )

    train_ir_ds, test_ir_ds = ir_ds.split(train_size=0.7, random_state=42)

    # 3. Simulate retrievals and set predictions
    train_ir_ds.set_predictions(simulate_retrievals(train_ir_ds.qrels))
    test_ir_ds.set_predictions(simulate_retrievals(test_ir_ds.qrels))

    # 4. Set embeddings for diagnostic suites
    train_ir_ds.set_embeddings(random_embedder)
    test_ir_ds.set_embeddings(random_embedder)

    return train_ir_ds, test_ir_ds

In [ ]:
# Load the dataset
# Subset to 10 queries for a super fast and light demonstration
train_data, test_data = load_ir_data(subset_queries=10)
print(f"Data loaded! Train pairs: {len(train_data)}, Test pairs: {len(test_data)}")

## Defining the IRLookupModel

In Information Retrieval and Entity Linking workflows, predictions (relevance scores, retrieval ranks) are usually pre-calculated by the retrieval engine (e.g., PyTerrier, BM25, or a bi-encoder).

DeepFix provides an `IRLookupModel` class that satisfies scikit-learn's estimator interface by looking up these pre-computed relevance predictions and probabilities for query-document pairs on the fly. This allows you to evaluate your pre-computed retrievals directly through our diagnostic pipeline.

In [ ]:
# Initialize lookup model
model = IRLookupModel(train_dataset=train_data, test_dataset=test_data)
model_name = "simulated_retrieval_model"

## Running DeepFix Diagnostic Analysis

Now we will run the automated diagnosis. The `DeepFixClient` will:
1. Ingest the datasets and the lookup model.
2. Trigger the Deepchecks diagnostic suites on the datasets to analyze metadata, data distribution, and text embeddings.
3. Call the model evaluator to measure precision, recall, and potential leakage or overfitting.
4. Synthesize all findings and provide a prioritized list of recommendations.

In [ ]:
# Run automated diagnosis
result = client.get_diagnosis(
    train_data=train_data,
    test_data=test_data,
    model=model,
    model_name=model_name,
    language="english"
)

## Reviewing Diagnostic Results

Once analysis is complete, we can visualize the prioritized findings and recommendations in plain text using the `.to_text()` method.

In [ ]:
# Display summary of findings and recommended actions
result.to_text()